# 03. Regularization - Ridge, Lasso, Elastic Net## What is this notebook about?When a linear model has too many features or its data has noise, it can **overfit** - learning noise instead of patterns. **Regularization** adds a penalty to the loss so the model can't go wild.Three flavours:- **Ridge (L2 penalty):** shrinks all coefficients gently toward zero- **Lasso (L1 penalty):** drives weak coefficients **exactly to zero** (automatic feature selection!)- **Elastic Net:** mix of L1 and L2## What you will learn1. Why plain Linear Regression overfits when you have many features2. How Ridge and Lasso shrink coefficients differently3. How to **visualize coefficient paths** as regularization strength changes4. How to **pick the best alpha** with cross-validation## DatasetCalifornia Housing again - same as notebook 01. We'll see what happens when we regularize.

In [ ]:
# If you're running locally, uncomment and run this once.# In Google Colab, all of these are pre-installed - you can skip this cell.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebookimport numpy as np                 # numerical arraysimport pandas as pd                # tabular data (DataFrames)import matplotlib.pyplot as plt    # plottingimport seaborn as sns              # prettier statistical plots# Set up plot stylesns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)# Set a random seed so results are reproduciblenp.random.seed(42)

In [ ]:
from sklearn.datasets import fetch_california_housingfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCVfrom sklearn.metrics import mean_squared_error, r2_score# Load the datadata = fetch_california_housing(as_frame=True)df = data.frameX = df.drop(columns=["MedHouseVal"])y = df["MedHouseVal"]X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)print(f"Training shape: {X_train.shape}")

## Step 1. Compare four models head-to-headWe'll train Linear, Ridge, Lasso, and Elastic Net using the same data and compare RMSE and R².

In [ ]:
def evaluate(name, model):    """Train one model in a Pipeline (with scaling) and print test scores."""    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])    pipe.fit(X_train, y_train)    pred = pipe.predict(X_test)    rmse = mean_squared_error(y_test, pred, squared=False)    r2 = r2_score(y_test, pred)    print(f"{name:18s}  RMSE = {rmse:.4f}   R^2 = {r2:.4f}")    return pipe# Compare modelsprint("Test set performance:")print("-" * 50)models = {    "Linear":       LinearRegression(),                          # no regularization    "Ridge":        Ridge(alpha=1.0),                           # L2 penalty    "Lasso":        Lasso(alpha=0.1),                           # L1 penalty    "Elastic Net":  ElasticNet(alpha=0.1, l1_ratio=0.5),        # both}fitted = {name: evaluate(name, m) for name, m in models.items()}

**On California Housing all four are similar** because the dataset isn't strongly affected by overfitting (only 8 features, 20k rows). Regularization shines more on datasets with **many features and few rows**.## Step 2. Visualize how coefficients shrink as alpha growsThis is the most beautiful illustration of how Ridge vs Lasso behave differently. We sweep `alpha` from very small (no regularization) to very large (heavy regularization).

In [ ]:
# Try 50 different alpha values, log-spaced from 0.001 to 1000alphas = np.logspace(-3, 3, 50)ridge_coefs = []lasso_coefs = []# Scale once, outside the loopscaler = StandardScaler().fit(X_train)X_train_s = scaler.transform(X_train)# For each alpha, fit Ridge and Lasso and record the coefficientsfor a in alphas:    r = Ridge(alpha=a).fit(X_train_s, y_train)    l = Lasso(alpha=a, max_iter=5000).fit(X_train_s, y_train)    ridge_coefs.append(r.coef_)    lasso_coefs.append(l.coef_)ridge_coefs = np.array(ridge_coefs)lasso_coefs = np.array(lasso_coefs)# Plot the two paths side by sidefig, axes = plt.subplots(1, 2, figsize=(14, 5))for i, name in enumerate(X.columns):    axes[0].plot(alphas, ridge_coefs[:, i], label=name)    axes[1].plot(alphas, lasso_coefs[:, i], label=name)for ax, title in zip(axes, ["Ridge (L2): smooth shrinkage", "Lasso (L1): exact zeros"]):    ax.set_xscale("log")    ax.set_xlabel("alpha (regularization strength)")    ax.set_ylabel("coefficient value")    ax.set_title(title)    ax.legend(fontsize=8)plt.tight_layout()plt.show()

**Key observation:**- **Ridge** (left): all coefficients smoothly shrink toward zero but never reach it.- **Lasso** (right): coefficients hit zero one by one as alpha grows. By alpha=10, only 1-2 features survive.This is why Lasso is great for **feature selection**.## Step 3. Pick the best alpha with cross-validationInstead of guessing, let scikit-learn try many alphas and pick the best.

In [ ]:
# RidgeCV and LassoCV try a list of alphas and select the one with best CV scoreridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50)).fit(X_train_s, y_train)lasso_cv = LassoCV(alphas=np.logspace(-3, 3, 50), cv=5, max_iter=5000).fit(X_train_s, y_train)print(f"Best Ridge alpha: {ridge_cv.alpha_:.5f}")print(f"Best Lasso alpha: {lasso_cv.alpha_:.5f}")# How many features did Lasso keep (non-zero coefficients)?n_kept = (lasso_cv.coef_ != 0).sum()print(f"\nLasso kept {n_kept} of {len(X.columns)} features")print("Selected features:", X.columns[lasso_cv.coef_ != 0].tolist())

## Step 4. Exercises1. **Generate noise features** to see Lasso filter them out:   ```python   X_noisy = X.copy()   for i in range(20):       X_noisy[f"noise_{i}"] = np.random.randn(len(X))   ```   Now train Lasso. How many noise features did it drop to zero?2. **Try `Lasso(alpha=10)`** - what's left of the model?3. **Plot RMSE vs alpha** for both Ridge and Lasso. Find the sweet spot visually.4. **Apply to a higher-dimensional dataset** like the Ames Housing competition on Kaggle - regularization makes a much bigger difference there.Next up: **04 - Decision Trees**!